# 🫀 PULSE ECG — Training (Direct Copy From Google Drive)
**Runtime → Change runtime type → GPU (T4)** before running!

In [ ]:
# 1️⃣ Mount Google Drive
# (If this cell fails with "credential propagation", try doing Runtime → Restart Session and try again)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2️⃣ Clone Project from GitHub
!git clone https://github.com/Gowtham-1301/TCN_BiLSTM.git /content/project
%cd /content/project

import os
for d in ['data/raw', 'data/processed', 'models/tfjs_model', 'logs', 'evaluation', 'checkpoints']:
    os.makedirs(d, exist_ok=True)
print('\n✅ Project code cloned successfully!')

In [ ]:
# 3️⃣ Copy Raw Data from Google Drive to Colab Local
# =========================================================================
# 👇 Make sure this path points to where data/raw is on YOUR Google Drive:
# =========================================================================
DRIVE_DATA_RAW = '/content/drive/MyDrive/pulse_ecg_model/data/raw'

import shutil
if os.path.exists(DRIVE_DATA_RAW):
    print(f'\n⏳ Copying data from Drive ({DRIVE_DATA_RAW}) to Local... This is much faster than downloading from PhysioNet!')
    os.system(f'cp -r "{DRIVE_DATA_RAW}"/* data/raw/')
    print('\n✅ Data completely copied to local!')
else:
    print(f'\n❌ ERROR: Could not find raw data at {DRIVE_DATA_RAW}. Please fix the path inside this cell.')

In [ ]:
# 4️⃣ Install Dependencies
!pip install -q wfdb scipy scikit-learn matplotlib seaborn tqdm PyYAML h5py pandas tensorboard
print('\n✅ Dependencies installed')

In [ ]:
# 5️⃣ Verify GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    !nvidia-smi
    print('\n✅ GPU is ready!')
else:
    print('\n❌ No GPU detected! Go to: Runtime → Change runtime type → GPU (T4)')

In [ ]:
# 6️⃣ Train the Model
# --skip-download forces the script to just look inside the currently filled data/raw directoy
!python train.py --skip-download

In [ ]:
# 7️⃣ View Results
import json, glob, os
from IPython.display import Image, display

for ef in sorted(glob.glob('evaluation/*.json')):
    print(f'\n📊 {os.path.basename(ef)}')
    with open(ef) as f:
        for k, v in json.load(f).items():
            if isinstance(v, float):
                print(f'  {k}: {v:.4f}')

for pf in sorted(glob.glob('evaluation/*.png')):
    display(Image(filename=pf, width=500))

In [ ]:
# 8️⃣ Export to TensorFlow.js
!pip install -q tensorflowjs
!python convert_to_tfjs.py --metadata
print('\n✅ Converted to TF.js format.')

In [ ]:
# ✅️9️⃣ Save Resulting Models directly back to Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/pulse_ecg_trained_models'
import shutil

print(f'\n✅ Saving models back to your drive at: {DRIVE_OUTPUT_DIR}')
if os.path.exists(DRIVE_OUTPUT_DIR):
    shutil.rmtree(DRIVE_OUTPUT_DIR)

shutil.copytree('models', DRIVE_OUTPUT_DIR)
print('\n🎉 Fully Copied! Check your Google Drive!')